# Segment B — Build the Tool-Use Loop by Hand
### Module 1: Claude API & Ecosystem Foundations

**Goal:** define two real tools (Wikipedia search, arXiv search), write the full
tool-use loop ourselves — no framework — and watch Claude choose which tool to use
based on the question.

This is the most important notebook of the day. Everything in Modules 2–4 is a
more convenient way of doing exactly what we're about to do by hand.

## 1. Setup

In [ ]:
import os
import json
import urllib.request
import urllib.parse
from anthropic import Anthropic

client = Anthropic()
MODEL = "claude-sonnet-4-6"

## 2. Define the actual tool functions

These are plain Python functions. Nothing Claude-specific about them yet — they
just hit a public API and return text. We'll use:

- **Wikipedia's Search API** — no key required
- **arXiv's API** — no key required

Both return XML/JSON we need to parse into something readable.

In [ ]:
def search_wikipedia(query: str, limit: int = 3) -> str:
    """Search Wikipedia and return short summaries of the top results."""
    # TODO: Build the Wikipedia search API URL using urllib.parse.urlencode with:
    #   action="query", list="search", srsearch=query, format="json", srlimit=limit
    # Base URL: "https://en.wikipedia.org/w/api.php?"
    #
    # Make the request with a User-Agent header: "Module1-ResearchCompanion/1.0"
    # Parse the JSON response.
    #
    # Extract results = data.get("query", {}).get("search", [])
    # If no results, return: f"No Wikipedia results found for '{query}'."
    #
    # For each result, strip HTML tags from the snippet:
    #   snippet.replace('<span class="searchmatch">', "").replace("</span>", "")
    # Build a list of "- {title}: {snippet}" strings and return them joined by "\n"
    pass


# Quick manual test — run this once to confirm the function itself works
# before handing it to Claude.
print(search_wikipedia("agentic AI"))

In [ ]:
def search_arxiv(query: str, max_results: int = 3) -> str:
    """Search arXiv and return titles, authors, and abstract snippets."""
    # TODO: Build the arXiv API URL using urllib.parse.urlencode with:
    #   search_query=f"all:{query}", start=0, max_results=max_results
    # Base URL: "http://export.arxiv.org/api/query?"
    #
    # Fetch the URL (no special headers needed for arXiv).
    # arXiv returns Atom XML — parse it with xml.etree.ElementTree:
    #   import xml.etree.ElementTree as ET
    #   ns = {"atom": "http://www.w3.org/2005/Atom"}
    #   root = ET.fromstring(raw)
    #   entries = root.findall("atom:entry", ns)
    #
    # If no entries, return: f"No arXiv results found for '{query}'."
    #
    # For each entry extract:
    #   title = e.find("atom:title", ns).text.strip().replace("\n", " ")
    #   authors = [a.find("atom:name", ns).text for a in e.findall("atom:author", ns)]
    #   summary = e.find("atom:summary", ns).text.strip().replace("\n", " ")[:200]
    # Build "- {title} ({', '.join(authors)}): {summary}..." and return joined by "\n"
    pass


# Manual test
print(search_arxiv("agentic retrieval"))

## 3. Describe these tools to Claude

A Python function isn't a "tool" to Claude until we describe it in the shape the
API expects: a `name`, a `description` (this is what Claude uses to *decide* whether
to call it — write it like documentation, not a label), and an `input_schema`
(JSON Schema) describing the arguments.

In [ ]:
tools = [
    {
        "name": "search_wikipedia",
        # TODO: Write a description that tells Claude when to use this tool
        # vs. search_arxiv. Think about: what kind of questions fit Wikipedia?
        # What doesn't? Your wording here directly affects which tool Claude picks.
        "description": "TODO: describe when Claude should use Wikipedia search",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query, e.g. a topic or concept name."
                }
            },
            "required": ["query"],
        },
    },
    {
        "name": "search_arxiv",
        # TODO: Write a description that tells Claude when to use arXiv
        # vs. Wikipedia. What makes a question better suited for academic papers?
        "description": "TODO: describe when Claude should use arXiv search",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query — topic, method name, or keywords."
                }
            },
            "required": ["query"],
        },
    },
]

## 4. One tool call, inspected by hand (before we automate it)

Let's send a question and a tool definition, and look at exactly what comes back —
**before** wrapping any of this in a loop. This is the slide from lecture made real.

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    tools=tools,
    messages=[
        {"role": "user", "content": "What's the latest research on agentic retrieval methods?"}
    ],
)

response

Look at `response.stop_reason` — it should be `"tool_use"`. And `response.content`
should contain a block with `type == "tool_use"`, a `name` matching one of our tools,
and an `input` dict with the arguments Claude generated.

This is exactly the round-trip from the lecture diagram. Claude hasn't *done*
anything yet — it's asked us to.

In [ ]:
for block in response.content:
    print(block.type)
    if block.type == "tool_use":
        print("  tool name:", block.name)
        print("  tool id:  ", block.id)
        print("  input:    ", block.input)

## 5. Execute the tool call and send the result back

Now we close the loop manually, once, so the mechanics are fully visible:
1. Map the tool name Claude asked for to our actual Python function
2. Call it with the arguments Claude provided
3. Package the result as a `tool_result` block
4. Send it back as a new message with `role: "user"`

In [ ]:
TOOL_FUNCTIONS = {
    "search_wikipedia": search_wikipedia,
    "search_arxiv": search_arxiv,
}

# Grab the tool_use block from the previous response
tool_use_block = next(b for b in response.content if b.type == "tool_use")

# Execute the actual function
function_to_call = TOOL_FUNCTIONS[tool_use_block.name]
result_text = function_to_call(**tool_use_block.input)

print(result_text)

In [ ]:
# Now send that result back. Note the conversation history so far:
# 1. our original user message
# 2. Claude's tool_use response
# 3. our tool_result, sent as role="user"

messages = [
    {"role": "user", "content": "What's the latest research on agentic retrieval methods?"},
    {"role": "assistant", "content": response.content},
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,
                "content": result_text,
            }
        ],
    },
]

final_response = client.messages.create(
    model=MODEL,
    max_tokens=500,
    tools=tools,
    messages=messages,
)

print(final_response.content[0].text)
print()
print("stop_reason:", final_response.stop_reason)

Notice the `tool_use_id` — it has to match the `id` from Claude's original
`tool_use` block exactly. This is how Claude matches your result back to its
request, especially important if it ever requests multiple tool calls at once.

## 6. Now automate it: the actual loop

We just did the loop manually, once. Real questions might need this to repeat —
Claude might want to call a second tool after seeing the first result, or decide
it has enough information and stop. Let's write a function that keeps going until
`stop_reason` is no longer `"tool_use"`.

In [ ]:
def run_agent(user_message: str, max_iterations: int = 5) -> str:
    """
    Run the full tool-use loop for a single user message.
    Returns Claude's final text answer once stop_reason == "end_turn".
    """
    # TODO: Implement the loop:
    #
    # 1. Start messages = [{"role": "user", "content": user_message}]
    #
    # 2. Loop up to max_iterations times:
    #    a. Call client.messages.create() with model, max_tokens=800, tools, messages
    #    b. If response.stop_reason != "tool_use":
    #         return response.content[0].text  (Claude is done)
    #    c. Otherwise:
    #       - Append {"role": "assistant", "content": response.content} to messages
    #       - Loop over response.content blocks; for each block where block.type == "tool_use":
    #           * Print: f"  [iteration {iteration}] calling {block.name}({block.input})"
    #           * Call TOOL_FUNCTIONS[block.name](**block.input)
    #           * Collect {"type": "tool_result", "tool_use_id": block.id, "content": output}
    #       - Append {"role": "user", "content": tool_results} to messages
    #
    # 3. If you exhaust max_iterations, return:
    #    "Reached max_iterations without a final answer — check for a loop."
    pass

## 7. Try it — and watch Claude choose

Ask something that should pull from Wikipedia, then something that should pull
from arXiv, then something ambiguous. Watch the printed tool calls to see which
one Claude picks and why — this is the first genuinely "agentic" moment of the
day: Claude is making a decision, not just following a script.

In [ ]:
print(run_agent("Give me a general overview of what reinforcement learning is."))

In [ ]:
print(run_agent("Find a recent paper specifically about multi-agent orchestration."))

In [ ]:
# An ambiguous one — see which tool Claude reaches for, and whether it uses both.
print(run_agent("What is Model Context Protocol, and is there academic research on it?"))

## Recap

- A "tool" is just a Python function plus a description Claude can read
- `description` is doing real work — it's how Claude decides *whether* and *which* tool to call
- The loop is: send messages → check `stop_reason` → if `tool_use`, execute + append result → repeat → return on `end_turn`
- Claude can request multiple tool calls in a single turn — our loop handles that by looping over every `tool_use` block, not just the first

Next: Segment C — turn this into a reusable, multi-turn agent and move it into a script.